In [2]:
import pandas as pd

path = "/Users/darshanv/clv-prediction/data/online_retail_II.csv"
df = pd.read_csv(path, encoding="ISO-8859-1")

print(df.shape)
print(df.columns.tolist())
df.head(10)

(1067371, 8)
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom


In [3]:
missing_customer = df["Customer ID"].isna().sum()
pct_missing = missing_customer / len(df) * 100

negative_qty = (df["Quantity"] < 0).sum()
pct_negative = negative_qty / len(df) * 100

cancelled_invoices = df["Invoice"].astype(str).str.startswith("C").sum()

print(f"Missing Customer ID: {missing_customer} rows ({pct_missing:.2f}%)")
print(f"Negative Quantity: {negative_qty} rows ({pct_negative:.2f}%)")
print(f"Rows with Invoice starting with 'C' (cancellations): {cancelled_invoices}")

Missing Customer ID: 243007 rows (22.77%)
Negative Quantity: 22950 rows (2.15%)
Rows with Invoice starting with 'C' (cancellations): 19494


In [4]:
df_clean = df.dropna(subset=["Customer ID"]).copy()
df_clean = df_clean[~df_clean["Invoice"].astype(str).str.startswith("C")]
df_clean = df_clean[df_clean["Quantity"] > 0]

print(f"Original rows: {len(df)}")
print(f"Rows after cleaning: {len(df_clean)}")
print(f"Rows dropped: {len(df) - len(df_clean)} ({(len(df)-len(df_clean))/len(df)*100:.2f}%)")

Original rows: 1067371
Rows after cleaning: 805620
Rows dropped: 261751 (24.52%)


In [5]:
df_clean["Customer ID"] = df_clean["Customer ID"].astype(int)
df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])
df_clean["LineTotal"] = df_clean["Quantity"] * df_clean["Price"]

orders = df_clean.groupby(["Invoice", "Customer ID", "InvoiceDate"]).agg(
    OrderValue=("LineTotal", "sum"),
    NumItems=("StockCode", "count")
).reset_index()

print(orders.shape)
orders.head()

(37039, 5)


,Invoice,Customer ID,InvoiceDate,OrderValue,NumItems
0,489434,13085,2009-12-01 07:45:00,505.30,8
1,489435,13085,2009-12-01 07:46:00,145.80,4
2,489436,13078,2009-12-01 09:06:00,630.33,19
3,489437,15362,2009-12-01 09:08:00,310.75,23
4,489438,18102,2009-12-01 09:24:00,2286.24,17


In [6]:
# count orders per customer
order_counts = orders.groupby("Customer ID").size().reset_index(name="NumOrders")

# how many customers only ordered once?
one_time = (order_counts["NumOrders"] == 1).sum()
total_customers = order_counts.shape[0]

print(f"Total customers: {total_customers}")
print(f"One-time customers: {one_time} ({one_time/total_customers*100:.2f}%)")

order_counts["NumOrders"].describe()

Total customers: 5881
One-time customers: 1623 (27.60%)


count    5881.000000
mean        6.298079
std        13.040416
min         1.000000
25%         1.000000
50%         3.000000
75%         7.000000
max       400.000000
Name: NumOrders, dtype: float64

In [7]:
# keep only customers with 2+ orders (need history to learn a pattern)
repeat_customers = order_counts[order_counts["NumOrders"] >= 2]["Customer ID"]
orders_filtered = orders[orders["Customer ID"].isin(repeat_customers)].copy()

print(f"Customers kept: {orders_filtered['Customer ID'].nunique()}")
print(f"Orders kept: {orders_filtered.shape[0]}")

# check distribution again to pick a sensible max sequence length
order_counts[order_counts["NumOrders"] >= 2]["NumOrders"].quantile([0.5, 0.75, 0.9, 0.95, 0.99])

Customers kept: 4258
Orders kept: 35416


0.50     5.00
0.75     9.00
0.90    17.00
0.95    25.00
0.99    55.43
Name: NumOrders, dtype: float64

In [8]:
# check overall date range to pick a sensible cutoff
print(orders_filtered["InvoiceDate"].min(), orders_filtered["InvoiceDate"].max())

2009-12-01 07:45:00 2011-12-09 12:50:00


In [9]:
cutoff_date = pd.Timestamp("2011-09-09")
future_end = cutoff_date + pd.Timedelta(days=90)

past_orders = orders_filtered[orders_filtered["InvoiceDate"] <= cutoff_date]
future_orders = orders_filtered[(orders_filtered["InvoiceDate"] > cutoff_date) & 
                                  (orders_filtered["InvoiceDate"] <= future_end)]

print(f"Past orders: {past_orders.shape[0]}, unique customers: {past_orders['Customer ID'].nunique()}")
print(f"Future orders: {future_orders.shape[0]}, unique customers: {future_orders['Customer ID'].nunique()}")

Past orders: 29092, unique customers: 4023
Future orders: 6174, unique customers: 2516


In [10]:
future_spend = future_orders.groupby("Customer ID")["OrderValue"].sum().reset_index(name="FutureSpend")

labels = past_orders[["Customer ID"]].drop_duplicates().merge(future_spend, on="Customer ID", how="left")
labels["FutureSpend"] = labels["FutureSpend"].fillna(0)

print(labels.shape)
labels["FutureSpend"].describe()

(4023, 2)


count      4023.000000
mean        707.404845
std        3648.924195
min           0.000000
25%           0.000000
50%         164.570000
75%         638.775000
max      112721.010000
Name: FutureSpend, dtype: float64

In [11]:
past_orders_sorted = past_orders.sort_values(["Customer ID", "InvoiceDate"]).copy()

# days since previous order, per customer
past_orders_sorted["DaysSinceLast"] = past_orders_sorted.groupby("Customer ID")["InvoiceDate"].diff().dt.days
past_orders_sorted["DaysSinceLast"] = past_orders_sorted["DaysSinceLast"].fillna(0)

# quick check on one customer to sanity-check the logic
sample_id = past_orders_sorted["Customer ID"].iloc[0]
past_orders_sorted[past_orders_sorted["Customer ID"] == sample_id][["Customer ID", "InvoiceDate", "OrderValue", "NumItems", "DaysSinceLast"]]

,Customer ID,InvoiceDate,OrderValue,NumItems,DaysSinceLast
1006,12346,2009-12-14 08:34:00,45.00,1,0.0
1017,12346,2009-12-14 11:00:00,22.50,1,0.0
1019,12346,2009-12-14 11:02:00,22.50,1,0.0
1361,12346,2009-12-18 10:47:00,22.50,1,3.0
1363,12346,2009-12-18 10:55:00,1.00,1,0.0
1513,12346,2010-01-04 09:24:00,22.50,1,16.0
1514,12346,2010-01-04 09:53:00,22.50,1,0.0
1834,12346,2010-01-14 13:50:00,22.50,1,10.0
2134,12346,2010-01-22 13:30:00,22.50,1,7.0
3708,12346,2010-03-02 13:08:00,27.05,5,38.0


In [12]:
# build one sequence (list of [DaysSinceLast, OrderValue, NumItems]) per customer
sequence_cols = ["DaysSinceLast", "OrderValue", "NumItems"]

customer_sequences = (
    past_orders_sorted.groupby("Customer ID")[sequence_cols]
    .apply(lambda x: x.values.tolist())
    .to_dict()
)

# sanity check: sequence length for our sample customer
print(f"Sequence length for customer {sample_id}: {len(customer_sequences[sample_id])}")
print(customer_sequences[sample_id][:3])  # first 3 steps


Sequence length for customer 12346: 12
[[0.0, 45.0, 1.0], [0.0, 22.5, 1.0], [0.0, 22.5, 1.0]]


In [14]:
import numpy as np

MAX_LEN = 20
NUM_FEATURES = 3  # DaysSinceLast, OrderValue, NumItems

def pad_sequence(seq, max_len=MAX_LEN):
    seq = seq[-max_len:]  # truncate to most recent `max_len` orders if longer
    pad_len = max_len - len(seq)
    padding = [[0.0, 0.0, 0.0]] * pad_len
    return padding + seq  # pad at start

padded_sequences = {
    cust_id: pad_sequence(seq) for cust_id, seq in customer_sequences.items()
}

# sanity check on our sample customer (12 orders, should have 8 padding rows at start)
sample_padded = padded_sequences[sample_id]
print(f"Padded length: {len(sample_padded)}")
for row in sample_padded:
    print(row)

Padded length: 20
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[0.0, 45.0, 1.0]
[0.0, 22.5, 1.0]
[0.0, 22.5, 1.0]
[3.0, 22.5, 1.0]
[0.0, 1.0, 1.0]
[16.0, 22.5, 1.0]
[0.0, 22.5, 1.0]
[10.0, 22.5, 1.0]
[7.0, 22.5, 1.0]
[38.0, 27.05, 5.0]
[118.0, 142.31, 19.0]
[203.0, 77183.6, 1.0]


In [15]:
customer_ids = list(padded_sequences.keys())

X = np.array([padded_sequences[cid] for cid in customer_ids])  # shape: (num_customers, 20, 3)
y = labels.set_index("Customer ID").loc[customer_ids]["FutureSpend"].values  # shape: (num_customers,)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X shape: (4023, 20, 3)
y shape: (4023,)


In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}, {y_train.shape}")
print(f"Test: {X_test.shape}, {y_test.shape}")

Train: (3218, 20, 3), (3218,)
Test: (805, 20, 3), (805,)


In [17]:
from sklearn.preprocessing import StandardScaler

# scale features: reshape to 2D for scaler, then back to 3D
scaler = StandardScaler()

n_samples, n_steps, n_features = X_train.shape
X_train_scaled = scaler.fit_transform(X_train.reshape(-1, n_features)).reshape(n_samples, n_steps, n_features)
X_test_scaled = scaler.transform(X_test.reshape(-1, n_features)).reshape(X_test.shape[0], n_steps, n_features)

# log-transform target (log1p handles zero values safely, unlike plain log)
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

print(X_train_scaled.mean(), X_train_scaled.std())
print(y_train_log[:5])

-2.679071429632033e-16 0.9999999999996615
[6.89310854 6.27884032 0.         5.71567811 5.62934624]


In [18]:
cutoff_date = pd.Timestamp("2011-09-09")

rfm = past_orders_sorted.groupby("Customer ID").agg(
    Recency=("InvoiceDate", lambda x: (cutoff_date - x.max()).days),
    Frequency=("InvoiceDate", "count"),
    Monetary=("OrderValue", "sum"),
    AvgOrderValue=("OrderValue", "mean")
).reset_index()

# align with same customer set and same train/test split as before
rfm = rfm.set_index("Customer ID").loc[customer_ids].reset_index()

print(rfm.shape)
rfm.head()

(4023, 5)


,Customer ID,Recency,Frequency,Monetary,AvgOrderValue
0,12346,233,12,77556.46,6463.038333
1,12347,37,6,4114.18,685.696667
2,12348,156,4,1709.40,427.350000
3,12349,315,3,2671.14,890.380000
4,12352,170,7,1905.61,272.230000


In [21]:
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

feature_cols = ["Recency", "Frequency", "Monetary", "AvgOrderValue"]

rfm["FutureSpend"] = y
rfm_train_df, rfm_test_df = train_test_split(rfm, test_size=0.2, random_state=42)

X_rfm_train = rfm_train_df[feature_cols]
X_rfm_test = rfm_test_df[feature_cols]
y_rfm_train_log = np.log1p(rfm_train_df["FutureSpend"])
y_rfm_test_log = np.log1p(rfm_test_df["FutureSpend"])

model = lgb.LGBMRegressor(random_state=42)
model.fit(X_rfm_train, y_rfm_train_log)

preds_log = model.predict(X_rfm_test)
preds = np.expm1(preds_log)
actual = rfm_test_df["FutureSpend"].values

mae = mean_absolute_error(actual, preds)
rmse = np.sqrt(mean_squared_error(actual, preds))
r2 = r2_score(actual, preds)

print(f"MAE: {mae:.2f}, RMSE: {rmse:.2f}, R2: {r2:.4f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000328 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 818
[LightGBM] [Info] Number of data points in the train set: 3218, number of used features: 4
[LightGBM] [Info] Start training from score 3.569651
MAE: 612.80, RMSE: 3547.15, R2: 0.3145
